**STEP 1 — Data preparation and LULC statistics**

In [1]:
#Install required packages
!pip -q install rasterio openpyxl pandas numpy matplotlib

In [2]:
# Import libraries
import os
import numpy as np
import pandas as pd
import rasterio
from google.colab import drive

In [4]:
# Mount Google Drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# Working directory
WORKDIR = "/content/drive/MyDrive/lulcmaps"

os.chdir(WORKDIR)

print("Working directory:")
print(os.getcwd())

Working directory:
/content/drive/MyDrive/lulcmaps


In [6]:
# Input rasters
FILES = {

    2020: "LULC2020.tif",
    2030: "LULC2030.tif",
    2040: "LULC2040.tif"

}

In [7]:
# LULC classes
CLASS_NAMES = {

    1: "Forest",
    2: "Rangeland",
    3: "Wetland",
    4: "Water body",
    5: "Cropland",
    6: "Built-up"

}

NODATA = 127

In [8]:
# Read rasters
rasters = {}

profiles = {}

pixel_area = None

for year, filename in FILES.items():

    with rasterio.open(filename) as src:

        rasters[year] = src.read(1)

        profiles[year] = src.profile

        transform = src.transform

        pixel_area = transform.a * abs(transform.e)

In [9]:
# Verify raster compatibility
shape = rasters[2020].shape
crs = profiles[2020]["crs"]

for year in [2030, 2040]:

    assert rasters[year].shape == shape, f"Shape mismatch for {year}"

    assert profiles[year]["crs"] == crs, f"CRS mismatch for {year}"

print("✓ Raster dimensions are identical.")

print("✓ Coordinate reference systems are identical.")

print(f"Raster size : {shape[0]} rows × {shape[1]} columns")

print(f"Pixel area : {pixel_area:.0f} m²")

✓ Raster dimensions are identical.
✓ Coordinate reference systems are identical.
Raster size : 6932 rows × 6855 columns
Pixel area : 900 m²


In [10]:
# Calculate LULC statistics
def calculate_statistics(raster):

    valid = raster[raster != NODATA]

    total_pixels = len(valid)

    results = []

    for cls in CLASS_NAMES:

        pixels = np.sum(valid == cls)

        area = pixels * pixel_area / 1e6

        percentage = pixels / total_pixels * 100

        results.append({

            "Class": cls,

            "Land use class": CLASS_NAMES[cls],

            "Pixels": pixels,

            "Area (km²)": round(area,2),

            "Area (%)": round(percentage,2)

        })

    return pd.DataFrame(results)

In [11]:
# Calculate statistics for all years
statistics = {}

for year in FILES:

    statistics[year] = calculate_statistics(rasters[year])

statistics[2020]

,Class,Land use class,Pixels,Area (km²),Area (%)
0,1,Forest,4574695,4117.23,30.24
1,2,Rangeland,5576431,5018.79,36.87
2,3,Wetland,166694,150.02,1.10
3,4,Water body,1961123,1765.01,12.97
4,5,Cropland,1753149,1577.83,11.59
5,6,Built-up,1094034,984.63,7.23


In [12]:
# Summary table
summary = pd.DataFrame({

    "Land use class": statistics[2020]["Land use class"],

    "2020 (km²)": statistics[2020]["Area (km²)"],

    "2030 (km²)": statistics[2030]["Area (km²)"],

    "2040 (km²)": statistics[2040]["Area (km²)"],

    "2020 (%)": statistics[2020]["Area (%)"],

    "2030 (%)": statistics[2030]["Area (%)"],

    "2040 (%)": statistics[2040]["Area (%)"]

})

summary

,Land use class,2020 (km²),2030 (km²),2040 (km²),2020 (%),2030 (%),2040 (%)
0,Forest,4117.23,4001.01,3930.45,30.24,9.36,9.19
1,Rangeland,5018.79,4556.43,4407.73,36.87,10.65,10.31
2,Wetland,150.02,127.94,124.70,1.10,0.30,0.29
3,Water body,1765.01,1775.86,1772.14,12.97,4.15,4.14
4,Cropland,1577.83,1950.05,2091.19,11.59,4.56,4.89
5,Built-up,984.63,1202.22,1287.30,7.23,2.81,3.01


In [14]:
# Export statistics
summary.to_csv("LULC_Area_Statistics_2020_2040.csv", index=False)

summary.to_excel("LULC_Area_Statistics_2020_2040.xlsx", index=False)

**STEP 2 — Land use transition analysis**

In [17]:
# Transition matrix function
def transition_matrix(map_from, map_to):

    valid = (map_from != NODATA) & (map_to != NODATA)

    from_pixels = map_from[valid]
    to_pixels = map_to[valid]

    matrix = np.zeros((6,6), dtype=np.int64)

    for i in range(1,7):
        for j in range(1,7):

            matrix[i-1, j-1] = np.sum(
                (from_pixels == i) &
                (to_pixels == j)
            )

    matrix = matrix * pixel_area / 1e6

    matrix = pd.DataFrame(
        matrix,
        index=list(CLASS_NAMES.values()),
        columns=list(CLASS_NAMES.values())
    )

    return matrix.round(2)

In [18]:
# Compute transition matrices
TM_2020_2030 = transition_matrix(
    rasters[2020],
    rasters[2030]
)

TM_2030_2040 = transition_matrix(
    rasters[2030],
    rasters[2040]
)
TM_2020_2030

,Forest,Rangeland,Wetland,Water body,Cropland,Built-up
Forest,4000.86,0.76,0.00,0.06,69.04,46.50
Rangeland,0.15,4553.86,0.00,1.59,298.55,164.63
Wetland,0.00,0.00,127.85,10.11,9.25,2.82
Water body,0.00,0.23,0.08,1764.08,0.09,0.53
Cropland,0.00,1.57,0.01,0.02,1573.12,3.11
Built-up,0.00,0.00,0.00,0.00,0.00,984.63


In [19]:
# Calculate persistence, gains and losses
def transition_statistics(matrix):

    persistence = np.diag(matrix)

    loss = matrix.sum(axis=1) - persistence

    gain = matrix.sum(axis=0) - persistence

    net = gain - loss

    stats = pd.DataFrame({

        "Persistence (km²)": persistence,

        "Loss (km²)": loss,

        "Gain (km²)": gain,

        "Net change (km²)": net

    }, index=matrix.index)

    return stats.round(2)

CHANGE_2020_2030 = transition_statistics(TM_2020_2030)

CHANGE_2030_2040 = transition_statistics(TM_2030_2040)

CHANGE_2020_2030

,Persistence (km²),Loss (km²),Gain (km²),Net change (km²)
Forest,4000.86,116.36,0.15,-116.21
Rangeland,4553.86,464.92,2.56,-462.36
Wetland,127.85,22.18,0.09,-22.09
Water body,1764.08,0.93,11.78,10.85
Cropland,1573.12,4.71,376.93,372.22
Built-up,984.63,0.00,217.59,217.59


In [20]:
# Prepare Sankey data
def create_sankey_table(matrix, year_from, year_to):

    rows = []

    for i, source_class in enumerate(matrix.index):

        for j, target_class in enumerate(matrix.columns):

            area = matrix.iloc[i, j]

            if area > 0:

                rows.append({

                    "source_year": year_from,

                    "target_year": year_to,

                    "source_class": source_class,

                    "target_class": target_class,

                    "area_km2": area

                })

    return pd.DataFrame(rows)

In [21]:
# Create Sankey dataset
Sankey_2020_2030 = create_sankey_table(
    TM_2020_2030,
    2020,
    2030
)

Sankey_2030_2040 = create_sankey_table(
    TM_2030_2040,
    2030,
    2040
)

Sankey_Data = pd.concat(
    [Sankey_2020_2030,
     Sankey_2030_2040],
    ignore_index=True
)

Sankey_Data.head()

,source_year,target_year,source_class,target_class,area_km2
0,2020,2030,Forest,Forest,4000.86
1,2020,2030,Forest,Rangeland,0.76
2,2020,2030,Forest,Water body,0.06
3,2020,2030,Forest,Cropland,69.04
4,2020,2030,Forest,Built-up,46.50


In [22]:
# Export transition analysis
TM_2020_2030.to_csv("TransitionMatrix_2020_2030.csv")
TM_2030_2040.to_csv("TransitionMatrix_2030_2040.csv")

CHANGE_2020_2030.to_csv("ChangeStatistics_2020_2030.csv")
CHANGE_2030_2040.to_csv("ChangeStatistics_2030_2040.csv")

Sankey_Data.to_csv("Sankey_Input_Data.csv", index=False)

**STEP 3 — Sankey Diagram**

In [23]:
# Install required libraries
!pip -q install plotly kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.6 MB/s eta 0:00:00


In [24]:
# Import libraries
import pandas as pd
import plotly.graph_objects as go

print("Libraries successfully imported.")

Libraries successfully imported.


In [25]:
# Read transition table
sankey = pd.read_csv("Sankey_Input_Data.csv")

print("Number of transitions:", len(sankey))

sankey.head()

Number of transitions: 57


,source_year,target_year,source_class,target_class,area_km2
0,2020,2030,Forest,Forest,4000.86
1,2020,2030,Forest,Rangeland,0.76
2,2020,2030,Forest,Water body,0.06
3,2020,2030,Forest,Cropland,69.04
4,2020,2030,Forest,Built-up,46.50


In [26]:
# Define Sankey node layout
classes = [

    "Forest",
    "Rangeland",
    "Wetland",
    "Water body",
    "Cropland",
    "Built-up"

]

years = [2020, 2030, 2040]

node_ids = []

for year in years:
    for cls in classes:
        node_ids.append(f"{cls}_{year}")

# Labels

node_labels = []

for year in years:
    for cls in classes:
        node_labels.append(cls)

# Fixed X positions
node_x = (

    [0.05] * 6 +      # 2020

    [0.50] * 6 +      # 2030

    [0.95] * 6        # 2040

)

# Fixed Y positions
y_positions = [

    0.05,   # Forest
    0.30,   # Rangeland
    0.50,   # Wetland
    0.65,   # Water body
    0.82,   # Cropland
    0.95    # Built-up

]

node_y = y_positions * 3

# LULC colors
class_colors = {

    "Forest": "#006400",
    "Rangeland": "#4BEB37",
    "Wetland": "#00C8C8",
    "Water body": "#1E88E1",
    "Cropland": "#F4E100",
    "Built-up": "#FF0000"

}

# Node colors
node_colors = []

for year in years:
    for cls in classes:
        node_colors.append(class_colors[cls])

print(f"Total nodes: {len(node_ids)}")

Total nodes: 18


In [27]:
# Build Sankey links
node_lookup = {
    node_id: index
    for index, node_id in enumerate(node_ids)
}

# Source class colors (RGBA with transparency)
flow_color_lookup = {

    "Forest": "rgba(0,100,0,0.45)",

    "Rangeland": "rgba(75,235,55,0.45)",

    "Wetland": "rgba(0,200,200,0.45)",

    "Water body": "rgba(30,136,225,0.45)",

    "Cropland": "rgba(244,225,0,0.45)",

    "Built-up": "rgba(255,0,0,0.45)"

}

# Lists required by Plotly
sources = []

targets = []

values = []

flow_colors = []

for _, row in sankey.iterrows():

    # Internal node IDs
    source_id = f"{row['source_class']}_{int(row['source_year'])}"
    target_id = f"{row['target_class']}_{int(row['target_year'])}"

    # Node indices
    sources.append(node_lookup[source_id])

    targets.append(node_lookup[target_id])

    # Transition area (km²)
    values.append(row["area_km2"])

    # Flow color
    flow_colors.append(
        flow_color_lookup[row["source_class"]]
    )
print(f"Total nodes : {len(node_ids)}")
print(f"Total links : {len(values)}")

Total nodes : 18
Total links : 57


In [29]:
# Create Sankey diagram

fig = go.Figure(

    go.Sankey(

        arrangement="fixed",

  # Nodes
            node=dict(

            pad=100,

            thickness=60,

            line=dict(
                color="black",
                width=1
            ),

            label=node_labels,

            color=node_colors,

            x=node_x,

            y=node_y,

            hovertemplate="%{label}<extra></extra>"

        ),

      # Links
            link=dict(

            source=sources,

            target=targets,

            value=values,

            color=flow_colors,

            hovertemplate=
            "<b>%{source.label}</b> → <b>%{target.label}</b><br>" +
            "Area: %{value:.2f} km²<extra></extra>"

        )

    )

)

In [30]:
# Layout, display and save
fig.update_layout(

    width=2244,
    height=1181,
    autosize=False,

    paper_bgcolor="white",
    plot_bgcolor="white",

    font=dict(
        family="Arial",
        size=36,
        color="black"
    ),

    margin=dict(
        l=0,
        r=0,
        t=120,
        b=0
    )
)

# Year labels
year_annotations = [

    (0.03, "2020"),
    (0.50, "2030"),
    (0.97, "2040")

]

for xpos, year in year_annotations:

    fig.add_annotation(

        x=xpos,
        y=1.1,

        xref="paper",
        yref="paper",

        text=f"<b>{year}</b>",

        showarrow=False,

        font=dict(
            family="Arial",
            size=36,
            color="black"
        )

    )

# Display figure
fig.show()

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.


